# Propensity Pipeline: Dedicated CatBoost Pipeline, Tuning & Calibration

This notebook implements a **production-grade, end-to-end ML pipeline** using the **CatBoost** algorithm to predict the probability (propensity score) of an HCP prescribing Pfizer.

### Pipeline Architecture
1. **Data Loading & Filtering** — Focus on high-potential B/C segments
2. **Target Definition & Anti-Leakage** — Broad target + automatic correlation audit
3. **Dimensionality Reduction** — Top-N feature selection to prevent overfitting using a fast CatBoost ranker
4. **Baseline CatBoost Model** — Stratified 5-Fold Cross-Validation baseline assessment
5. **Optuna Hyperparameter Optimization** — Bayesian search specifically on CatBoost
6. **Probability Calibration** — Isotonic calibration for trustworthy propensity scores
7. **Glass-Box Explainability** — SHAP global + local reason codes per HCP


In [1]:
# =====================================================================
# BLOCK 1: Imports and Environment Setup
# =====================================================================
# WHY: A clean, reproducible environment is the foundation of any
# trustworthy ML pipeline. We import all dependencies upfront so
# failures surface immediately, not halfway through a 30-minute run.
# =====================================================================

import pandas as pd
import numpy as np
import catboost as cb
import optuna
import shap
import matplotlib.pyplot as plt
import joblib
import warnings
import re
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectFromModel
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, average_precision_score,
    precision_recall_curve, roc_curve, make_scorer
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Paths ---
# Data lives in the shared Xgboost_probabilities/data folder.
# We read from there directly to avoid data duplication.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# CatBoost patch for scikit-learn compatibility
class PatchedCatBoostClassifier(cb.CatBoostClassifier):
    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.estimator_type = "classifier"
        return tags

print(" All imports successful.")
print(f"   CatBoost: {cb.__version__}")
print(f"   Optuna:   {optuna.__version__}")
print(f"   SHAP:     {shap.__version__}")


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# =====================================================================
# BLOCK 2: Data Loading & Segment Identification
# =====================================================================
# WHY: We load ALL HCPs and train on the full dataset to maximize the
# amount of training data available. The B/C segment HCPs (633) are
# identified and will be used exclusively as the TEST set for evaluation
# and final scoring.
# =====================================================================

print("Loading data...")

# Read the feature matrix (all HCPs) and the segment labels
features = pd.read_parquet(DATA_DIR / 'hcp_feature_matrix.parquet')
labels   = pd.read_csv(DATA_DIR / 'test_predictions_binary_segA_vs_segBC_with_hcp_id.csv')

# Ensure consistent ID types for the merge
features['NUEVO_ID'] = features['NUEVO_ID'].astype(str)
labels['NUEVO_ID']   = labels['NUEVO_ID'].astype(str)

# Inner join: only keep HCPs that exist in both datasets
df = features.merge(labels, on='NUEVO_ID', how='inner', suffixes=('', '_lbl'))

# Create a boolean mask identifying B/C segment HCPs (our TEST set)
bc_mask = (df['pred_binary_label'] == 'SEG_BC').values  # numpy array to persist after column drops

print(f"Dataset loaded. Shape: {df.shape}")
print(f"   Total HCPs (all segments): {len(df)}")
print(f"   B/C segment HCPs (test set): {bc_mask.sum()}")
print(f"   Non-B/C HCPs (train only): {(~bc_mask).sum()}")
print(f"   Total features available:   {df.shape[1]}")


Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\Xgboost_probabilities\\data\\hcp_feature_matrix.parquet'

In [ ]:
# =====================================================================
# BLOCK 3: Target Definition & Anti-Leakage Protection
# =====================================================================
# WHY: Data leakage is the #1 silent killer of ML models in production.
# If a feature directly encodes the target (e.g., "has_prescribed_brand1"),
# the model will appear perfect in training but fail catastrophically
# on new data. We implement a TWO-LAYER defense:
#   Layer 1: Explicit drop of known brand/metadata columns
#   Layer 2: Automated Pearson correlation audit (threshold > 0.90)
#
# SPLIT STRATEGY: Train on ALL HCPs, test/score on B/C segment only.
# This maximizes training data while focusing evaluation on the segment
# that matters for business decisions.
# =====================================================================

# --- Target Definition ---
# The target captures whether an HCP has EVER prescribed the Pfizer brand.
# This is the best proxy for "propensity to prescribe" given our data.
TARGET_COL = 'propensity_target'
df[TARGET_COL] = (df['BRAND1_TRX__max'] > 0).astype(int)

pos_count = df[TARGET_COL].sum()
neg_count = len(df) - pos_count
print(f"Target Distribution (ALL HCPs):")
print(f"  Negative (0): {neg_count}")
print(f"  Positive (1): {pos_count}")
print(f"  Positive prevalence: {pos_count / len(df) * 100:.2f}%\n")

# Save HCP IDs for final output (before dropping them from features)
hcp_ids = df['NUEVO_ID'].copy()

# --- LAYER 1: Explicit Leakage Prevention ---
# Drop ALL brand-related columns (they directly encode the target)
brand_cols = [c for c in df.columns if 'BRAND1_' in c]

# Drop metadata/label columns that would leak segment or model info
metadata_cols = [
    'ATSEG_HCP', 'IS_LABELED_HCP', 'HCP_FOLD', 'n_rows',
    'NUEVO_ID', 'NUEVO_ID.1', 'true_original_label', 'true_original_encoded',
    'true_binary_label', 'true_binary_encoded', 'prob_SEG_A', 'prob_SEG_BC',
    'pred_binary_label', 'pred_binary_encoded', 'decision_threshold',
    'hcp_fold', 'model_name', 'true_label_encoded', TARGET_COL
]
# Also drop any columns that came from the labels merge (suffix '_lbl')
metadata_cols += [c for c in df.columns if c.endswith('_lbl')]

cols_to_drop = list(set(brand_cols + metadata_cols))
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

X = df.drop(columns=cols_to_drop)
X = X.select_dtypes(include=[np.number])  # Keep only numeric features
y = df[TARGET_COL]

print(f"[LAYER 1] Dropped {len(cols_to_drop)} known leaky/metadata columns.")
print(f"          Remaining features: {X.shape[1]}")

# Fix invalid column names (XGBoost/LightGBM don't allow [, ], <)
X.columns = [re.sub(r'[\[\]<]', '_', c) for c in X.columns]

# --- LAYER 2: Automated Correlation Audit ---
print("\n[LAYER 2] Scanning features for correlations > 0.90 with target...")
correlations = X.corrwith(y).abs()
leaky_features = correlations[correlations > 0.90].index.tolist()

if leaky_features:
    print(f"  WARNING: Found {len(leaky_features)} leaky features. Dropping:")
    for feat in leaky_features:
        print(f"     - {feat} (corr={correlations[feat]:.4f})")
    X = X.drop(columns=leaky_features)
else:
    print("  No hidden leakage detected.")

# --- Handle missing values ---
# Fill NaNs with 0 (appropriate for sparse medical claim features)
missing_before = X.isnull().sum().sum()
X = X.fillna(0)
print(f"\n[PREPROCESSING] Filled {missing_before} missing values with 0.")

# --- Train/Test Split ---
# TRAIN: ALL HCPs (full dataset) — maximize training data
# TEST:  Only the 633 B/C segment HCPs — the segment we want to score
X_train = X.copy()
y_train = y.copy()

X_test = X[bc_mask].copy()
y_test = y[bc_mask].copy()

# Track which HCP IDs are in the test set (B/C segment)
hcp_ids_test = hcp_ids[bc_mask].reset_index(drop=True)

print(f"\nData ready.")
print(f"   Train: {X_train.shape} | Positive: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"   Test (B/C): {X_test.shape}  | Positive: {y_test.sum()} ({y_test.mean()*100:.1f}%)")


In [ ]:
# =====================================================================
# BLOCK 4: Automated Feature Selection (Dimensionality Reduction)
# =====================================================================
# WHY: With ~624 features and only 633 samples, we're at high risk of
# overfitting. Feature selection reduces noise, speeds up training, and
# produces more interpretable models. We use a lightweight CatBoost
# estimator to rank features by importance, then keep the top 40.
# =====================================================================

MAX_FEATURES = 40  # Balances signal vs. overfitting for this sample size

print(f"Selecting top {MAX_FEATURES} features using CatBoost to prevent overfitting...")

# Use a quick CatBoost model to rank feature importance
fs_model = PatchedCatBoostClassifier(
    random_state=RANDOM_STATE,
    verbose=0,
    iterations=100  # Fast, just for ranking
)
fs = SelectFromModel(fs_model, max_features=MAX_FEATURES, threshold=-np.inf)
fs.fit(X_train, y_train)

selected_features = X_train.columns[fs.get_support()].tolist()
X_train_sel = X_train[selected_features]
X_test_sel  = X_test[selected_features]

print(f"✅ Reduced feature space: {X_train.shape[1]} → {X_train_sel.shape[1]} features.")
print(f"\nSelected Features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i:2d}. {feat}")


In [ ]:
# =====================================================================
# BLOCK 5: CatBoost Baseline Evaluation
# =====================================================================
# WHY: We evaluate the baseline CatBoost model using Stratified 5-Fold
# Cross-Validation to set our benchmark performance before tuning.
# =====================================================================

pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"Class imbalance weight (Balanced): {pos_weight:.2f}\n")

model = PatchedCatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    random_state=RANDOM_STATE,
    verbose=0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_scores = []

print("Evaluating baseline CatBoost Model (5-Fold CV)... ")
for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_train_sel, y_train)):
    X_tr, X_val = X_train_sel.iloc[tr_idx], X_train_sel.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    model_clone = model.__class__(**model.get_params())
    model_clone.fit(X_tr, y_tr)
    preds = model_clone.predict_proba(X_val)[:, 1]
    score = average_precision_score(y_val, preds)
    fold_scores.append(score)

baseline_score = np.mean(fold_scores)
baseline_std  = np.std(fold_scores)
print(f"Baseline CatBoost PR-AUC = {baseline_score:.4f} ± {baseline_std:.4f}")


In [ ]:
# =====================================================================
# BLOCK 6: Optuna Hyperparameter Tuning
# =====================================================================
# WHY: Default hyperparameters leave performance on the table. Optuna's
# Bayesian optimization intelligently explores the search space,
# finding better configs in fewer trials.
# =====================================================================

N_OPTUNA_TRIALS = 30

print(f"Tuning CatBoost with Optuna ({N_OPTUNA_TRIALS} trials)...")
print(f"   Optimizing: PR-AUC (Average Precision) via 5-Fold CV\n")

def objective(trial):
    model = PatchedCatBoostClassifier(
        iterations=trial.suggest_int('iterations', 200, 1000),
        depth=trial.suggest_int('depth', 3, 9),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.15),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        bagging_temperature=trial.suggest_float('bagging_temperature', 0.0, 5.0),
        random_strength=trial.suggest_float('random_strength', 0.0, 5.0),
        auto_class_weights='Balanced',
        random_state=RANDOM_STATE,
        verbose=0
    )
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    
    for tr_idx, val_idx in cv_inner.split(X_train_sel, y_train):
        X_tr, X_val = X_train_sel.iloc[tr_idx], X_train_sel.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        try:
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_tr, y_tr)
            preds = model_clone.predict_proba(X_val)[:, 1]
            scores.append(average_precision_score(y_val, preds))
        except Exception:
            pass
    
    return np.mean(scores) if scores else 0.0

# --- Run Optuna ---
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nOptuna Optimization Complete!")
print(f"   Best PR-AUC: {study.best_value:.4f}")
print(f"   Best Trial:  #{study.best_trial.number}")
print(f"\n   Best Hyperparameters:")
for k, v in study.best_params.items():
    print(f"     {k}: {v}")


In [ ]:
# =====================================================================
# BLOCK 7: Probability Calibration (Isotonic)
# =====================================================================
# WHY: Raw model outputs are NOT true probabilities. Calibration using
# isotonic regression adjusts the output to be a TRUE probability estimate.
# =====================================================================

print("Training tuned model and applying isotonic calibration...\n")

# --- Retrain the tuned model on the full training set ---
best_model = PatchedCatBoostClassifier(
    iterations=study.best_params['iterations'],
    depth=study.best_params['depth'],
    learning_rate=study.best_params['learning_rate'],
    l2_leaf_reg=study.best_params['l2_leaf_reg'],
    bagging_temperature=study.best_params['bagging_temperature'],
    random_strength=study.best_params['random_strength'],
    auto_class_weights='Balanced',
    random_state=RANDOM_STATE,
    verbose=0
)

best_model.fit(X_train_sel, y_train)

# --- Evaluate BEFORE calibration ---
y_proba_uncal = best_model.predict_proba(X_test_sel)[:, 1]
prauc_uncal   = average_precision_score(y_test, y_proba_uncal)
rocauc_uncal  = roc_auc_score(y_test, y_proba_uncal)
brier_uncal   = brier_score_loss(y_test, y_proba_uncal)

print(f"BEFORE Calibration:")
print(f"  ROC-AUC:    {rocauc_uncal:.4f}")
print(f"  PR-AUC:     {prauc_uncal:.4f}")
print(f"  Brier Score: {brier_uncal:.4f}\n")

# --- Apply Isotonic Calibration ---
calibrated_model = CalibratedClassifierCV(
    best_model, method='isotonic', cv=5
)
calibrated_model.fit(X_train_sel, y_train)

# --- Evaluate AFTER calibration ---
y_proba_cal = calibrated_model.predict_proba(X_test_sel)[:, 1]
prauc_cal   = average_precision_score(y_test, y_proba_cal)
rocauc_cal  = roc_auc_score(y_test, y_proba_cal)
brier_cal   = brier_score_loss(y_test, y_proba_cal)

print(f"AFTER Calibration (Isotonic):")
print(f"  ROC-AUC:    {rocauc_cal:.4f}")
print(f"  PR-AUC:     {prauc_cal:.4f}")
print(f"  Brier Score: {brier_cal:.4f}")

# --- Summary ---
print(f"\nCalibration Impact:")
print(f"   Brier Score: {brier_uncal:.4f} → {brier_cal:.4f} ({'✅ Improved' if brier_cal < brier_uncal else '⚠️  Slightly worse'})")
print(f"   PR-AUC:     {prauc_uncal:.4f} → {prauc_cal:.4f}")


In [ ]:
# --- Reliability Diagram (Calibration Curve) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Calibration Curve
ax = axes[0]
for label, proba, color in [
    ('Before Calibration', y_proba_uncal, '#e74c3c'),
    ('After Calibration (Isotonic)', y_proba_cal, '#2ecc71')
]:
    fraction_pos, mean_predicted = calibration_curve(y_test, proba, n_bins=8)
    ax.plot(mean_predicted, fraction_pos, 's-', label=label, color=color, linewidth=2)

ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfectly Calibrated')
ax.set_xlabel('Mean Predicted Probability', fontsize=11)
ax.set_ylabel('Fraction of Positives', fontsize=11)
ax.set_title('Reliability Diagram', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: Score Distribution
ax = axes[1]
ax.hist(y_proba_cal[y_test == 0], bins=30, alpha=0.6, label='Negative', color='#3498db', density=True)
ax.hist(y_proba_cal[y_test == 1], bins=30, alpha=0.6, label='Positive', color='#e74c3c', density=True)
ax.set_xlabel('Calibrated Probability', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Score Distribution by Class', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'calibration_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Calibration analysis saved to {OUTPUT_DIR / 'calibration_analysis.png'}")


In [ ]:
# =====================================================================
# BLOCK 8: SHAP Glass-Box Explainability & Final Export
# =====================================================================
# WHY: A black-box propensity score is useless if stakeholders can't
# understand WHY a particular HCP has a high/low score. SHAP provides:
#   - GLOBAL explanations: which features matter most overall
#   - LOCAL explanations: for each HCP, the top 5 reasons driving
#     their specific score ("reason codes")
# =====================================================================

print("Computing SHAP explanations...\n")

# --- Use TreeExplainer on the UNCALIBRATED model ---
explainer = shap.TreeExplainer(best_model)

# We compute SHAP values for ALL 633 B/C segment HCPs (the test set)
X_bc_sel = X_test_sel  # B/C segment with selected features
shap_values = explainer.shap_values(X_bc_sel)

# Handle different SHAP output formats depending on the model
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Take the positive class

print(f"   SHAP values computed for {shap_values.shape[0]} B/C segment HCPs.")
print(f"   SHAP matrix shape: {shap_values.shape}")


In [ ]:
# --- Global SHAP Summary Plot ---
print("\nGlobal Feature Importance (SHAP Beeswarm Plot):")
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_bc_sel, plot_type="dot", show=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_global_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"   Saved to: {OUTPUT_DIR / 'shap_global_summary.png'}")


In [ ]:
# --- Extract Top 5 Local Reason Codes per HCP ---

def extract_top_k_reasons(shap_vals_row, feature_values_row, feature_names, k=5):
    """
    For a single instance, extract the top-k most impactful features
    (by absolute SHAP value), along with their SHAP contribution and
    raw feature value.
    
    Returns:
        dict with keys: reason_1..reason_k, reason_1_shap..reason_k_shap,
                        reason_1_value..reason_k_value
    """
    # Sort by absolute SHAP value (most impactful first)
    sorted_idx = np.argsort(np.abs(shap_vals_row))[::-1][:k]
    
    result = {}
    for rank, idx in enumerate(sorted_idx, 1):
        result[f'reason_{rank}']       = feature_names[idx]
        result[f'reason_{rank}_shap']  = round(float(shap_vals_row[idx]), 6)
        result[f'reason_{rank}_value'] = round(float(feature_values_row[idx]), 4)
    
    return result

print("Extracting Top 5 Reason Codes for each B/C segment HCP...")

feature_names = X_bc_sel.columns.tolist()
reason_records = []

for i in range(len(X_bc_sel)):
    reasons = extract_top_k_reasons(
        shap_vals_row=shap_values[i],
        feature_values_row=X_bc_sel.iloc[i].values,
        feature_names=feature_names,
        k=5
    )
    reason_records.append(reasons)

reasons_df = pd.DataFrame(reason_records)
print(f"✅ Extracted reason codes for {len(reasons_df)} B/C segment HCPs.")


In [ ]:
# --- Assemble Final Prediction DataFrame ---
# Compute calibrated propensity scores for ALL 633 B/C segment HCPs
y_proba_cal_bc = calibrated_model.predict_proba(X_bc_sel)[:, 1]

final_df = pd.DataFrame({
    'NUEVO_ID': hcp_ids_test.values,
    'propensity_score': y_proba_cal_bc,
    'actual_label': y_test.values
})

# Append reason codes
final_df = pd.concat([final_df.reset_index(drop=True), reasons_df], axis=1)

# Sort by propensity score (highest first = most likely prescribers)
final_df = final_df.sort_values('propensity_score', ascending=False).reset_index(drop=True)

print("\n Final Prediction DataFrame (Top 10 HCPs):")
print("=" * 120)
display_cols = ['NUEVO_ID', 'propensity_score', 'actual_label',
                'reason_1', 'reason_1_shap', 'reason_2', 'reason_2_shap']
print(final_df[display_cols].head(10).to_string(index=False))

# --- Export to Parquet ---
output_path = OUTPUT_DIR / 'propensity_predictions_with_reasons.parquet'
final_df.to_parquet(output_path, index=False)
print(f"\n💾 Final predictions exported to: {output_path}")
print(f"   Shape: {final_df.shape}")
print(f"   Columns: {final_df.columns.tolist()}")


In [ ]:
# =====================================================================
# FINAL SUMMARY
# =====================================================================

print("\n" + "=" * 65)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 65)
print(f"")
print(f"Model Algorithm:       CatBoost (Dedicated Pipeline)")
print(f"    Baseline PR-AUC:       {baseline_score:.4f}")
print(f"")
print(f"🔧 Optuna Best PR-AUC:     {study.best_value:.4f} ({N_OPTUNA_TRIALS} trials)")
print(f"")
print(f"Final Test Metrics (Calibrated, on B/C segment):")
print(f"    ROC-AUC:               {rocauc_cal:.4f}")
print(f"    PR-AUC:                {prauc_cal:.4f}")
print(f"    Brier Score:           {brier_cal:.4f}")
print(f"")
print(f"Output Files:")
print(f"    Predictions:  {OUTPUT_DIR / 'propensity_predictions_with_reasons.parquet'}")
print(f"    Calibration:  {OUTPUT_DIR / 'calibration_analysis.png'}")
print(f"    SHAP Summary: {OUTPUT_DIR / 'shap_global_summary.png'}")
print(f"")
print(f"Training: ALL {len(X_train)} HCPs | Testing: {len(final_df)} B/C segment HCPs")
print(f"{len(final_df)} B/C segment HCPs scored with Top 5 SHAP reason codes.")
print("=" * 65)
